In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( uti )
importlib.reload( puti )
importlib.reload( nuti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)


In [ ]:
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")


In [ ]:
path1='/glade/derecho/scratch/aherring/archive/newclubb_FHISTC_MTso_ne30pg3_ne30pg3_mg17_top_lev_bug/atm/hist/newclubb_FHISTC_MTso_ne30pg3_ne30pg3_mg17_top_lev_bug.cam.h0a.*.nc'
X1 = xr.open_mfdataset( path1 ,data_vars='different', coords='different', compat='no_conflicts' )

Q1 = X1['Q']
Q1_anncycle = Q1.groupby('time.month').mean('time')

In [ ]:
path2='/glade/derecho/scratch/aherring/archive/newclubb_FHISTC_MTso_ne30pg3_ne30pg3_mg17_top_lev_fix/atm/hist/newclubb_FHISTC_MTso_ne30pg3_ne30pg3_mg17_top_lev_fix.cam.h0a.*.nc'
X2 = xr.open_mfdataset( path2 ,data_vars='different', coords='different', compat='no_conflicts' )

Q2 = X2['Q']
Q2_anncycle = Q2.groupby('time.month').mean('time')

In [ ]:
Q2_anncycle

In [ ]:
basedir=f'/glade/campaign/cgd/amp/amwg/climo'

In [ ]:


case1={'name': f'f.e30_alpha08b.FHISTC_MTso.ne30_t232_wgx3.315' , 'years':[1980,1995] }
case2={'name': f'f.e30_alpha08b.FHISTC_MTso.ne30_t232_wgx3.316' , 'years':[1980,1995] }
case3={'name': f'f.e30_cam6_4_097.FHISTC_MTso.ne30.001', 'years':[1990,2004] }
fld='Q'


cases=[case3,case1,case2 ]


In [ ]:
"""
date_=f"{year:04d}-{month:02d}-{day:02d}-{hour*3_600:05d}"
/glade/campaign/cgd/amp/amwg/climo/f.e30_cam6_4_097.FHISTC_MTso.ne30.001/yrs_1990_2004
f.e30_alpha08b.FHISTC_MTso.ne30_t232_wgx3.316_Q_climo.nc
"""

In [ ]:
Al=[]
for case in cases:
    casename=case['name']
    y0_ = f"{case['years'][0]}"
    y1_ = f"{case['years'][1]}"
    filename = f"{basedir}/{casename}/yrs_{y0_}_{y1_}/{casename}_{fld}_climo.nc"
    print( filename )
    X=xr.open_dataset( filename )
    A_={'name':casename,'years':case['years'], 'X':X }
    A=AttrDict( A_ )
    Al.append( A )
    

In [ ]:
Al[0].X

In [ ]:
mos=np.arange( start=1, stop=13 )


In [ ]:
print( mos )

In [ ]:
n=0
q=Al[n].X.Q.values
lev=Al[n].X.lev.values
lat=Al[n].X.lat.values
lon=Al[n].X.lon.values

zlev=-7_000. * np.log( lev /1_000. )
print(zlev)

In [ ]:
ADF_dir = '/glade/campaign/cgd/amp/amwg/ADF_obs'
MLS = xr.open_dataset( f"{ADF_dir}/mls_h2o_latNpressNtime_3d_monthly_v5.nc" )
QERA = xr.open_dataset( f"{ADF_dir}/Q_ERA5_monthly_climo_1degree_197901-202112.nc" )

In [ ]:
levo=QERA.lev.values
qo=QERA.Q.values
zlevo=-7_000. * np.log( levo /1_000. )
print(levo)

In [ ]:
qo.shape

In [ ]:
Qs=[]
Qs.append( Q1_anncycle )
Qs.append( Q2_anncycle )


In [ ]:

nplots=len(Qs)
names = ['Buggy','Fixed',]
if nplots <= 5:
    nxp=nplots + 1
    nyp=1

leves=1.e-6 *np.linspace( 1.5 , 3.5 , num=21 )

xsize=7.
x_figsize , y_figsize =xsize*nxp , 0.99*xsize*nyp

fix,axs = plt.subplots( nyp, nxp, figsize=(x_figsize , y_figsize) )

ip=0
for Qx in Qs:    
    q=Qx.values
    lev=Qx.lev.values
    lat=Qx.lat.values
    lon=Qx.lon.values
    zlev=-7_000. * np.log( lev /1_000. )
    ax=axs[ip] 
    co = ax.contourf( mos, zlev, np.mean( q[:,:,95,:] , axis=2 ).T ,levels=leves, cmap='managua')
    ax.set_ylim(15_000.,60_000.)
    ax.set_xlabel('Month')
    ax.set_ylabel('Z (m)')
    ax.set_title( names[ip] )
    ip=ip+1
    cb = plt.colorbar(co, ax=ax, orientation='horizontal')
    cb.set_label("Water Vapor (kg kg$^{-1}$)")

ax=axs[ip] 
co = ax.contourf( mos, zlevo, np.mean( qo[:,:,90,:] , axis=2 ).T ,levels=leves, cmap='managua')
ax.set_ylim(15_000.,60_000.)
ax.set_xlabel('Month')
ax.set_ylabel('Z (m)')
ax.set_title( 'ERA5' )
cb = plt.colorbar(co, ax=ax, orientation='horizontal')
cb.set_label("Water Vapor (kg kg$^{-1}$)")


In [ ]:

nplots=len(Al)

if nplots <= 5:
    nxp=nplots + 1
    nyp=1

leves=1.e-6 *np.linspace( 1.5 , 3.5 , num=21 )

xsize=7.
x_figsize , y_figsize =xsize*nxp , 0.99*xsize*nyp

fix,axs = plt.subplots( nyp, nxp, figsize=(x_figsize , y_figsize) )

ip=0
for A in Al:    
    q=A.X.Q.values
    lev=A.X.lev.values
    lat=A.X.lat.values
    lon=A.X.lon.values

    ax=axs[ip] 
    co = ax.contourf( mos, zlev, np.mean( q[:,:,95,:] , axis=2 ).T ,levels=leves, cmap='managua')
    ax.set_ylim(15_000.,60_000.)
    ax.set_xlabel('Month')
    ax.set_ylabel('Z (m)')
    ax.set_title( A.name )
    ip=ip+1
    cb = plt.colorbar(co, ax=ax, orientation='horizontal')
    cb.set_label("Water Vapor (kg kg$^{-1}$)")

ax=axs[ip] 
co = ax.contourf( mos, zlevo, np.mean( qo[:,:,90,:] , axis=2 ).T ,levels=leves, cmap='managua')
ax.set_ylim(15_000.,60_000.)
ax.set_xlabel('Month')
ax.set_ylabel('Z (m)')
ax.set_title( 'ERA5' )
cb = plt.colorbar(co, ax=ax, orientation='horizontal')
cb.set_label("Water Vapor (kg kg$^{-1}$)")


In [ ]:
print( np.argmin( np.abs(zlev-17000) ) )
print( np.argmin( np.abs(zlevo-17000) ) )
z = np.argmin( np.abs(zlev-17000) ) 
zo = np.argmin( np.abs(zlevo-17000) ) 


In [ ]:


fix,axs = plt.subplots( 1, 2, figsize=(23 , 6) )

ip=0
ax=axs[ip]
z_p = 17_000.
z = np.argmin( np.abs(zlev-z_p) ) 
zo = np.argmin( np.abs(zlevo-z_p) ) 
for A in Al:    
    q=A.X.Q.values
    ax.plot( mos, np.mean( q[:,:,95,:],axis=2 )[:,z] , label=A.name)
    
ax.plot( mos, np.mean( qo[:,:,95,:],axis=2 )[:,zo] , label='ERA5')
ax.set_title( f'Z={z_p:05.0f} m',fontsize=20)
ax.set_xlabel('Month',fontsize=18)
ax.set_ylabel("Water Vapor (kg kg$^{-1}$)",fontsize=18)
ax.legend()

ip=1
ax=axs[ip]
z_p = 50_000.
z = np.argmin( np.abs(zlev-z_p) ) 
zo = np.argmin( np.abs(zlevo-z_p) ) 
for A in Al:    
    q=A.X.Q.values
    ax.plot( mos, np.mean( q[:,:,95,:],axis=2 )[:,z] , label=A.name)
    
ax.plot( mos, np.mean( qo[:,:,95,:],axis=2 )[:,zo] , label='ERA5')
ax.set_title( f'Z={z_p:05.0f} m',fontsize=20)
ax.set_xlabel('Month',fontsize=18)
ax.set_ylabel("Water Vapor (kg kg$^{-1}$)",fontsize=18)
ax.legend()

In [ ]:


fix,axs = plt.subplots( 1, 2, figsize=(23 , 6) )

ip=0
ax=axs[ip]
z_p = 17_000.
z = np.argmin( np.abs(zlev-z_p) ) 
zo = np.argmin( np.abs(zlevo-z_p) ) 
for A in Al:    
    q=A.X.Q.values
    ax.plot( mos, np.mean( q[:,:,95,:],axis=2 )[:,z] , label=A.name)
    
ax.plot( mos, np.mean( qo[:,:,95,:],axis=2 )[:,zo] , label='ERA5')
ax.set_title( f'Z={z_p:05.0f} m',fontsize=20)
ax.set_xlabel('Month',fontsize=18)
ax.set_ylabel("Water Vapor (kg kg$^{-1}$)",fontsize=18)
ax.legend()

ip=1
ax=axs[ip]
z_p = 50_000.
z = np.argmin( np.abs(zlev-z_p) ) 
zo = np.argmin( np.abs(zlevo-z_p) ) 
for A in Al:    
    q=A.X.Q.values
    ax.plot( mos, np.mean( q[:,:,95,:],axis=2 )[:,z] , label=A.name)
    
ax.plot( mos, np.mean( qo[:,:,95,:],axis=2 )[:,zo] , label='ERA5')
ax.set_title( f'Z={z_p:05.0f} m',fontsize=20)
ax.set_xlabel('Month',fontsize=18)
ax.set_ylabel("Water Vapor (kg kg$^{-1}$)",fontsize=18)
ax.legend()